In [2]:
from base64 import b64encode
from pathlib import Path

from dotenv import load_dotenv

from thesis.config import load_config

load_dotenv()

models = {entry["provider"]: entry for entry in load_config("models")["models"]}


image_path = Path("/Users/phillip/repositories/thesis-vlm-worktree/data/file.png")
image_b64 = b64encode(image_path.read_bytes()).decode()

mime_type = "image/png"
prompt = "Does the pin in the center of the image lie on the roof of the building? Answer yes or no."

FileNotFoundError: [Errno 2] No such file or directory: '/Users/phillip/repositories/thesis-vlm-worktree/data/file.png'

In [ ]:
from openai import OpenAI

cfg = models["openai"]
params = cfg["parameters"]

response = OpenAI().responses.create(
    model=cfg["model"],
    reasoning=params["reasoning"],
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:{mime_type};base64,{image_b64}",
                    "detail": params["detail"],
                },
            ],
        }
    ],
)

print(response.output_text)
print(response.usage)

Yes.
ResponseUsage(input_tokens=336, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=6, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=342)


In [ ]:
from anthropic import Anthropic

cfg = models["anthropic"]
mime_type = "image/png"

message = Anthropic().messages.create(
    model=cfg["model"],
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": mime_type,
                        "data": image_b64,
                    },
                },
                {"type": "text", "text": prompt},
            ],
        }
    ],
)

print(message)
print(message.usage)

NameError: name 'image_b64' is not defined

In [ ]:
import os

from anthropic import Anthropic

load_dotenv()
cfg = models["anthropic"]

message = Anthropic().messages.create(
    workspace_id=os.getenv("ANTHROPIC_WORKSPACE_ID"),
    model=cfg["model"],
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Say hi back"},
            ],
        }
    ],
)

print(message)
print(message.usage)

Message(id='msg_011CemmdLfNypTz82Rqqu4CB', container=None, content=[TextBlock(citations=None, text="Hi there! 👋 What's on your mind today?", type='text')], model='claude-opus-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=11, output_tokens=19, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=11, output_tokens=19, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard')


In [1]:
from google import genai

from thesis.config import load_config

models = {entry["provider"]: entry for entry in load_config("models")["models"]}

client = genai.Client()
config = models["google"]
params = config["parameters"]

interaction = client.interactions.create(
    model=config["model"],
    input=[
        {"type": "text", "text": "hallo"},
    ],
    generation_config=params["thinking_config"],
)

# hm
print(interaction.output_text)
print(interaction.usage)

Hallo! Wie kann ich dir heute helfen?
cached_tokens_by_modality=None grounding_tool_count=None input_tokens_by_modality=[ModalityTokens(modality='text', tokens=3)] output_tokens_by_modality=None tool_use_tokens_by_modality=None total_cached_tokens=0 total_input_tokens=3 total_output_tokens=9 total_thought_tokens=156 total_tokens=168 total_tool_use_tokens=0 raw_prompt_token=34 model_invocation_token_counts=[{'prompt_tokens_details': [{'modality': 'text', 'tokens': 34}], 'candidates_tokens_details': [{'modality': 'text', 'tokens': 13}], 'thoughts_tokens_details': [{'modality': 'text', 'tokens': 156}]}]


In [ ]:
from thesis.vlm.run import classify

image = Path("/Users/phillip/repositories/thesis-vlm-worktree/data/file.png").read_bytes()
# prompt = "Is the pin correctly placed on the target? Answer yes or no."

results = classify(image, prompt)

# for provider, result in results.items():
#     print(
#         provider,
#         result.parsed_answer,
#         result.raw_text,
#         result.cost,
#     )

results["openai"].model_dump()
results["anthropic"].model_dump()
results["google"].model_dump()

{'provider': 'google',
 'raw_text': 'Yes',
 'parsed_answer': 'yes',
 'requested_model_id': 'gemini-3.6-flash',
 'returned_model_id': 'gemini-3.6-flash',
 'prompt_id': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'prompt_sha256': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'image_sha256': '503e8acf67775aeadc34ec1654007cc6c7933beafa00883590e68b6397c22c67',
 'complete_request_settings': {'model': 'gemini-3.6-flash',
  'generation_config': {'thinking_level': 'low'},
  'image': {'mime_type': 'image/png', 'resolution': 'high'}},
 'provider_request_id': 'v1_ChdhMGVjYW9mLU9zV2ZrZFVQbjZfNnNBWRIXYTBlY2FvZi1Pc1dma2RVUG42XzZzQVk',
 'input_token_count': 1112,
 'image_token_count': 1089,
 'image_token_count_kind': 'provider_reported',
 'output_token_count': 1,
 'thinking_token_count': 149,
 'latency': 4.182836208026856,
 'price_schedule': {'tier': 'standard',
  'image_px': 512,
  'currency': 'USD',
  'as_of': '2026-09-04',
  'input': 0.75,
  'output':

In [ ]:
results["anthropic"].model_dump()

{'provider': 'anthropic',
 'raw_text': 'No',
 'parsed_answer': 'no',
 'requested_model_id': 'claude-opus-5',
 'returned_model_id': 'claude-opus-5',
 'prompt_id': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'prompt_sha256': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'image_sha256': '503e8acf67775aeadc34ec1654007cc6c7933beafa00883590e68b6397c22c67',
 'complete_request_settings': {'model': 'claude-opus-5',
  'max_tokens': 1024,
  'thinking': {'type': 'adaptive'},
  'output_config': {'effort': 'low'},
  'image': {'media_type': 'image/png'},
  'workspace_id': 'wrkspc_01JbkpDF1TwH7dZiGL37X3NH'},
 'provider_request_id': 'msg_011CekdbWK4yWHK3hKAM5Vst',
 'input_token_count': 399,
 'image_token_count': 361,
 'image_token_count_kind': 'exact',
 'output_token_count': 3,
 'thinking_token_count': 0,
 'latency': 9.823673874954693,
 'price_schedule': {'tier': 'standard',
  'image_px': 512,
  'currency': 'USD',
  'as_of': '2026-09-04',
  'input': 5.0,


In [ ]:
results["openai"].model_dump()

{'provider': 'openai',
 'raw_text': 'Yes.',
 'parsed_answer': 'yes',
 'requested_model_id': 'gpt-5.6-sol',
 'returned_model_id': 'gpt-5.6-sol',
 'prompt_id': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'prompt_sha256': 'a448aadda90f8c20750ad4ffeebd47ab1cadd3479336d451c8b64f50171082f6',
 'image_sha256': '503e8acf67775aeadc34ec1654007cc6c7933beafa00883590e68b6397c22c67',
 'complete_request_settings': {'model': 'gpt-5.6-sol',
  'reasoning': {'effort': 'low'},
  'image': {'mime_type': 'image/png', 'detail': 'original'}},
 'provider_request_id': 'resp_00a72afb45df469f006a9c475ea88c87d29b047cc2fde829cb',
 'input_token_count': 336,
 'image_token_count': 308,
 'image_token_count_kind': 'estimate',
 'output_token_count': 6,
 'thinking_token_count': 0,
 'latency': 4.789398625027388,
 'price_schedule': {'tier': 'standard',
  'image_px': 512,
  'currency': 'USD',
  'as_of': '2026-09-04',
  'input': 4.0,
  'output': 20.0},
 'cost': 0.001464,
 'timestamp': 1788626778.885997,